# Advanced tutorial: Export a reproducible research subset

## Research question

**How can a researcher create and share the exact approved temperature subset used to study summer conditions in a selected HUC8?**

## `streamwq` functions used

- `StreamClient.find_sites()`
- `StreamClient.read_water_quality()`
- `filter_season()`
- `resample_timeseries()`
- `export_subset()`

In [ ]:
from streamwq.client import StreamClient
from streamwq.export import export_subset
from streamwq.temporal import filter_season, resample_timeseries

stream = StreamClient()

## 1. Record the query in one place

Keeping query choices in a dictionary makes them visible in the notebook and allows the same information to be written to the provenance manifest. Replace the example HUC8 with a code selected in the HUC map tutorial if desired.

In [ ]:
query = {
    "huc8": "02070010",
    "variable": "WTemp_C",
    "quality": "approved",
    "start": "2018-01-01",
    "end": "2022-12-31",
    "season": "summer",
    "frequency": "daily",
    "statistic": "mean",
}
query

## 2. Build the subset

In [ ]:
sites = stream.find_sites(
    huc8=query["huc8"], variables=query["variable"]
)
site_ids = sites["STREAM_ID"].tolist()
observations = stream.read_water_quality(
    site_ids=site_ids,
    variables=query["variable"],
    start=query["start"],
    end=query["end"],
    quality=query["quality"],
)
summer = filter_season(observations, query["season"])
daily = resample_timeseries(
    summer, query["variable"], query["frequency"], query["statistic"]
)
daily.head()

## 3. Export data and provenance

The export creates a Parquet file and a neighboring JSON file containing the HydroShare resource, package version, query, dimensions, columns, and export time.

In [ ]:
data_path, metadata_path = export_subset(
    daily,
    "outputs/huc8-summer-temperature.parquet",
    query=query,
    description="Approved daily mean summer water temperature for one HUC8.",
)
print(data_path)
print(metadata_path)

In [ ]:
import json

print(json.dumps(json.loads(metadata_path.read_text()), indent=2))